# STRATA, três vertentes: Llama-3.1-8B contra k-quants com imatrix

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*.

As três vertentes de `docs/07-quantizacao-aninhada.md`, medidas na mesma sessão, com a mesma imatrix e a régua decidida antes de rodar:

1. **Escala conjunta** (`strata_base_fina.py --conjunta A`): a escala de cada sub-bloco do Q4 é escolhida pensando também no erro dos níveis 1 e 2. O nível 0 continua um Q4_K comum e os níveis usam os tipos de hoje. Varre o peso `A` dos níveis de cima.
2. **Base fina T=1 com nível 2** (`strata_base_fina.py --T 1 --K 2`): a que já ganhou no nível 1 (KL 7,59 contra 8,60 do Q5_K_M).
3. **Sub-células não uniformes** (`strata_escala_nivel.py --modo desloc`): o nível 0 é o Q4_K_M com imatrix intacto, byte a byte; em cada nível, os dois filhos de cada célula ganham deslocamentos próprios por código, ajustados com a imatrix. Ainda sem kernel: medida por emulação em F16.
4. **Combinação 1 + 3** (`V13`): o nível 0 da escala conjunta com `A_COMBO`, e as sub-células não uniformes em cima.

No Llama-3.2-1B (container, KL ×1000; k-quants com imatrix: Q4_K_M 40,9, Q5_K_M 13,9, Q6_K 7,3 no caminho real; Q5_K_M 11,8 e Q6_K 5,4 em F16):
- escala conjunta A = 0: 40,3 / 17,5 / 6,3; A = 0,25: 42,6 / 15,7 / 5,8; A = 1: 47,3 / 14,2 / 5,6 (níveis 0 / 1 / 2);
- base fina T=1 com nível 2: 54,2 / 12,5 / 6,1;
- sub-células por linha (F16): nível 0 = Q4_K_M, 11,7 / 3,5, com +0,07 e +0,22 bit/peso de tabelas; por tensor: 13,1 / 4,1, sem custo.

**Dois caminhos de medida.** As vertentes 1 e 2 rodam nos tipos do STRATA no CUDA (caminho real) e são comparadas com os k-quants normais. A 3 roda em F16, onde as ativações não são arredondadas para 8 bits e a KL de todos cai; por isso ela é comparada com os k-quants desquantizados para F16, e o STRATA de hoje em F16 entra como referência. Compare cada vertente só com a curva do seu caminho.

**Régua:** KL de cada nível contra a curva dos k-quants com imatrix (Q3_K_M, Q4_K_M, Q5_K_M, Q6_K e Q8_0) no mesmo tamanho de RAM; custo do nível 0 em bits por peso; velocidade de geração e prompt no caminho real; desempate: nível 0 idêntico ao Q4_K_M. A última célula monta a tabela da régua.

Ocupa ~60 GB de disco no pico (cada GGUF é apagado depois de medido). A parte mais lenta é montar os pesos na CPU.

In [ ]:
A_CONJUNTA = [0.5, 1.0, 2.0]   # pesos dos níveis de cima na escala conjunta (0 = só o nível 0)
A_COMBO = 1.0              # nível 0 da escala conjunta com esse A + sub-células por linha em cima (None = não medir)
DESLOC = ['linha', 'tensor']   # grupos da vertente 3: deslocamentos por linha (melhor, ~0,05 bit) e por tensor (~0 bit)
KLD_CHUNKS = 100           # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125       # trechos de 512 tokens do wikitext de treino para a imatrix
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100 (ou L4).'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 75:
    print('Aviso: pouco disco. Ponha KLD_CHUNKS menor.')
if RAM < 50:
    print('Aviso: pouca RAM para a vertente 3. Ligue a RAM alta.')

In [ ]:
BRANCH = "claude/project-thread-qjksmf"
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh strata/kernel/ggml/aplicar.sh llama.cpp
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
# o fim do log da compilação fica visível, para aparecer qualquer erro
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-perplexity llama-imatrix llama-bench 2>&1 | tail -15
!ls -la llama.cpp/build/bin | grep -E 'llama-(perplexity|quantize|imatrix|bench)$'

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste,', os.path.getsize('/content/wiki.train.raw'), 'de treino')

Modelo em BF16, imatrix, k-quants com imatrix e a base da KL. Os pesos originais são apagados depois da conversão (tudo lê o GGUF BF16).

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!rm -rf {src}
!llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
for q in ['Q3_K_M', 'Q4_K_M', 'Q5_K_M', 'Q6_K']:
    !llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {G}/BF16.gguf {G}/{q}_imat.gguf {q} > {G}/{q}_imat.log 2>&1
!llama.cpp/build/bin/llama-quantize {G}/BF16.gguf {G}/Q8_0.gguf Q8_0 > {G}/Q8_0.log 2>&1
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
!ls -la {G}/*.gguf /content/kld_base.bin
!df -h /content | tail -1

Funções de medida. `tam` guarda o tamanho de cada GGUF antes de ele ser apagado.

In [ ]:
kld, tam, vel, tabelas = {}, {}, {}, {}
def kl_de(path, nome):
    out = !llama.cpp/build/bin/llama-perplexity -m {path} --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {nome}:', '\n'.join(out[-8:]))
        return
    kld[nome] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    tam[nome] = os.path.getsize(path) / 2**30
    print(f'{nome}: KL ×1000 {1000 * kld[nome][0]:.2f}, mesmo 1º token {kld[nome][2]}%, {tam[nome]:.3f} GiB')

def bench(path, nome):
    """tok/s de geração (tg128) e de prompt (pp512) na GPU, 3 repetições."""
    out = !llama.cpp/build/bin/llama-bench -m {path} -ngl 99 -p 512 -n 128 -r 3 -o json 2>/dev/null
    try:
        r = json.loads('\n'.join(l for l in out if not l.startswith('ggml_') and not l.startswith('load')))
        vel[nome] = {('pp' if x['n_prompt'] else 'tg'): x['avg_ts'] for x in r}
        print(f'{nome}: {vel[nome]}')
    except Exception as e:
        print(f'Erro no llama-bench de {nome}:', e, '\n'.join(out[-5:]))

Caminho real: k-quants com imatrix (KL e velocidade).

In [ ]:
for q in ['Q3_K_M_imat', 'Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q8_0']:
    kl_de(f'{G}/{q}.gguf', q)
for q in ['Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat']:
    bench(f'{G}/{q}.gguf', q)
!rm -f {G}/Q3_K_M_imat.gguf

Vertente 2: base fina T=1 com nível 2 (caminho real, KL e velocidade).

In [ ]:
FINA = f'python strata/tests/kernel/strata_base_fina.py --bf16 {G}/BF16.gguf --gguf {G}/Q4_K_M_imat.gguf --imatrix {G}/imatrix.gguf'
!{FINA} --T 1 --K 2 --out {G}/V2_N0.gguf {G}/V2_N1.gguf {G}/V2_N2.gguf 2>&1 | grep -v Warning | tail -3
for k in range(3):
    kl_de(f'{G}/V2_N{k}.gguf', f'V2_N{k}')
bench(f'{G}/V2_N1.gguf', 'STRATA_N1')
bench(f'{G}/V2_N2.gguf', 'STRATA_N2')
!rm -f {G}/V2_N*.gguf

Vertente 1: escala conjunta, um conjunto de níveis por valor de `A` (caminho real, só KL: os tipos são os mesmos da vertente 2).

In [ ]:
for A in A_CONJUNTA:
    n = f'V1_A{A:g}'
    !{FINA} --T 2 --conjunta {A} --out {G}/{n}_N0.gguf {G}/{n}_N1.gguf {G}/{n}_N2.gguf 2>&1 | grep -v Warning | tail -3
    for k in range(3):
        kl_de(f'{G}/{n}_N{k}.gguf', f'{n}_N{k}')
    !rm -f {G}/{n}_N1.gguf {G}/{n}_N2.gguf
    if A != A_COMBO:
        !rm -f {G}/{n}_N0.gguf

Vertente 3 e referências, caminho F16: k-quants desquantizados, o STRATA de hoje (passo fixo) e as sub-células não uniformes. Cada GGUF F16 tem ~16 GB.

In [ ]:
ESC = f'python strata/tests/kernel/strata_escala_nivel.py --bf16 {G}/BF16.gguf --imatrix {G}/imatrix.gguf'
for q in ['Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q8_0']:
    !{ESC} --gguf {G}/{q}.gguf --modo copia --out /content/e.gguf 2>&1 | grep -v Warning | tail -1
    kl_de('/content/e.gguf', 'F16_' + q)
    tam['F16_' + q] = tam[q]
    !rm -f /content/e.gguf
    if q != 'Q4_K_M_imat':
        !rm -f {G}/{q}.gguf
!{ESC} --gguf {G}/Q4_K_M_imat.gguf --modo fixo --out /content/e1.gguf /content/e2.gguf 2>&1 | grep -v Warning | grep -v '^ *vals'
kl_de('/content/e1.gguf', 'F16_fixo_N1')
kl_de('/content/e2.gguf', 'F16_fixo_N2')
!rm -f /content/e1.gguf /content/e2.gguf
for g in DESLOC:
    out = !{ESC} --gguf {G}/Q4_K_M_imat.gguf --modo desloc --grupo {g} --out /content/e1.gguf /content/e2.gguf 2>&1
    out = [l for l in out if 'Warning' not in l]
    print('\n'.join(out[-3:]))
    for k in (1, 2):
        kl_de(f'/content/e{k}.gguf', f'V3_{g}_N{k}')
        m = re.search(rf'nível {k}, .*tabelas de ([0-9.]+) MiB', '\n'.join(out))
        tabelas[f'V3_{g}_N{k}'] = float(m.group(1)) if m else float('nan')
    !rm -f /content/e1.gguf /content/e2.gguf
if A_COMBO is not None:                                   # vertentes 1 + 3: base da escala conjunta, sub-células em cima
    out = !{ESC} --gguf {G}/V1_A{A_COMBO:g}_N0.gguf --modo desloc --grupo linha --out /content/e1.gguf /content/e2.gguf 2>&1
    out = [l for l in out if 'Warning' not in l]
    for k in (1, 2):
        kl_de(f'/content/e{k}.gguf', f'V13_N{k}')
        m = re.search(rf'nível {k}, .*tabelas de ([0-9.]+) MiB', '\n'.join(out))
        tabelas[f'V13_N{k}'] = float(m.group(1)) if m else float('nan')
    !rm -f /content/e1.gguf /content/e2.gguf {G}/V1_A{A_COMBO:g}_N0.gguf

**A régua.** Para cada nível de cada vertente:
- *RAM*: o tamanho do GGUF no caminho real. Na vertente 3, o tamanho do STRATA de hoje no mesmo nível mais as tabelas. Na coluna "Q6_K empacotado", o nível 1 sem o byte inteiro por peso nas matrizes Q6_K (7,56 em vez de 8,56 bits/peso), que é o que um tipo empacotado daria.
- *k-quant no mesmo tamanho*: interpolação do log da KL entre os k-quants do mesmo caminho, no tamanho do candidato.
- *razão*: KL do candidato dividida pela do k-quant interpolado (abaixo de 1 = o STRATA ganha no mesmo tamanho).
- *nível 0*: quantos bits por peso o nível 0 "perde", pela curva entre Q3_K_M e Q4_K_M.

In [ ]:
import math, gguf
from IPython.display import Markdown, display
# pesos das matrizes das camadas que são Q6_K no Q4_K_M (para o tamanho com o Q6_K empacotado)
r = gguf.GGUFReader(f'{G}/Q4_K_M_imat.gguf')
n_q6 = sum(int(t.n_elements) for t in r.tensors if t.name.startswith('blk.') and t.tensor_type.name == 'Q6_K' and t.name.endswith('.weight') and t.n_elements > 1e6)
n_cam = sum(int(t.n_elements) for t in r.tensors if t.name.startswith('blk.') and t.tensor_type.name in ('Q4_K', 'Q6_K') and t.n_elements > 1e6)
EMP = n_q6 / 8 / 2**30                     # GiB a menos no nível 1 com o Q6_K empacotado

def curva(pref, s):
    """KL do k-quant interpolado no tamanho s (GiB), no caminho pref ('' real, 'F16_' emulado)."""
    pts = sorted((tam[pref + q], kld[pref + q][0]) for q in ['Q3_K_M_imat', 'Q4_K_M_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q8_0'] if pref + q in kld)
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if s <= s2 or (s2, k2) == pts[-1]:
            if s < pts[0][0]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return k1 * math.exp(math.log(k2 / k1) * (s - s1) / (s2 - s1))

def bits_n0(nome):
    """bits por peso a menos que o nível 0 'vale', pela curva entre Q3_K_M e Q4_K_M."""
    s1, k1, s2, k2 = tam['Q3_K_M_imat'], kld['Q3_K_M_imat'][0], tam['Q4_K_M_imat'], kld['Q4_K_M_imat'][0]
    s_eq = s1 + (s2 - s1) * math.log(kld[nome][0] / k1) / math.log(k2 / k1)
    return (tam[nome] - s_eq) * 2**30 * 8 / n_cam

linhas = []
def regua(nome, nivel, pref='', s=None, emp=False):
    if nome not in kld:
        return
    s = tam[nome] if s is None else s
    k = kld[nome][0]
    if nivel == 0 and pref == '':
        linhas.append(f'| {nome} | 0 | {s:.3f} | | {1000 * k:.2f} | {1000 * kld["Q4_K_M_imat"][0]:.2f} (Q4_K_M) | {k / kld["Q4_K_M_imat"][0]:.2f} | {bits_n0(nome):+.3f} |')
        return
    c = curva(pref, s)
    txt = f'| {nome} | {nivel} | {s:.3f} | '
    if emp:
        se = s - EMP
        txt += f'{se:.3f} → razão {k / curva(pref, se):.2f} | '
    else:
        txt += '| '
    txt += f'{1000 * k:.2f} | {1000 * c:.2f} | {k / c:.2f} | |'
    linhas.append(txt)

L = [f'# STRATA, três vertentes (Llama-3.1-8B, llama.cpp CUDA)', '',
     f'GPU: {GPU}. KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos do {TEXTO} de teste; imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. '
     f'Matrizes das camadas: {n_cam / 1e9:.2f} bilhões de pesos, {n_q6 / 1e9:.2f} bilhões em Q6_K.', '',
     '## KL e tamanho', '', '| modelo | GGUF (GiB) | KL (×1000) | mesmo 1º token |', '|---|---:|---:|---:|']
for m, v in kld.items():
    L.append(f'| {m} | {tam[m]:.3f} | {1000 * v[0]:.2f} ± {1000 * v[1]:.2f} | {v[2]:.2f}% |')

L += ['', '## Régua (KL no mesmo tamanho de RAM; razão abaixo de 1 = STRATA melhor)', '',
      '| candidato | nível | RAM (GiB) | Q6_K empacotado | KL ×1000 | k-quant no mesmo tamanho | razão | nível 0: bits/peso a menos |',
      '|---|---:|---:|---|---:|---:|---:|---:|']
for k in range(3):
    regua(f'V2_N{k}', k, emp=(k == 1))
for A in A_CONJUNTA:
    for k in range(3):
        regua(f'V1_A{A:g}_N{k}', k, emp=(k == 1))
for k in (1, 2):
    regua(f'F16_fixo_N{k}', k, 'F16_', s=tam.get(f'V2_N{k}'), emp=(k == 1))
    for pre in [f'V3_{g}' for g in DESLOC] + ['V13']:
        nome = f'{pre}_N{k}'
        if f'V2_N{k}' in tam and nome in kld:
            tab = sum(tabelas.get(f'{pre}_N{j}', 0) for j in range(1, k + 1))   # o nível k precisa das tabelas dos níveis 1 … k
            regua(nome, k, 'F16_', s=tam[f'V2_N{k}'] + tab / 1024, emp=(k == 1))
L += linhas
L += ['', 'Vertente 3 e F16_fixo: caminho F16, RAM = a do STRATA no caminho real (mesmos tipos) mais as tabelas dos níveis até ele; curva = k-quants em F16. '
      f'O nível 0 da vertente 3 é o próprio Q4_K_M_imat; o da V13 é o V1_A{A_COMBO:g}_N0.']

if vel:
    L += ['', '## Velocidade na GPU (tok/s; relação com o k-quant de mesma qualidade)', '', '| modelo | geração (tg128) | prompt (pp512) |', '|---|---:|---:|']
    for m, v in vel.items():
        L.append(f'| {m} | {v.get("tg", float("nan")):.1f} | {v.get("pp", float("nan")):.0f} |')
    for a, b in [('STRATA_N1', 'Q5_K_M_imat'), ('STRATA_N2', 'Q6_K_imat')]:
        if a in vel and b in vel:
            L.append(f'| {a} / {b} | {vel[a]["tg"] / vel[b]["tg"]:.2f} | {vel[a]["pp"] / vel[b]["pp"]:.2f} |')

os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_vertentes.md', 'w').write('\n'.join(L) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS, 'kld': kld, 'gguf_gib': tam,
           'vel': vel, 'tabelas_mib': tabelas, 'a_conjunta': A_CONJUNTA, 'n_q6': n_q6, 'n_camadas': n_cam},
          open('/content/res/strata_8b_vertentes.json', 'w'), indent=1)
display(Markdown('\n'.join(L)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_vertentes.md')
files.download('/content/res/strata_8b_vertentes.json')